In [3]:
import openwakeword
import numpy as np
import wave
import contextlib
import os
import tqdm
import warnings
import logging
import pandas as pd

# Tắt toàn bộ cảnh báo từ Python và thư viện
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

def test(model_path, model_name, test_file_path, start_end=[0,10000], threshold=0.2):
    oww = openwakeword.Model(
        wakeword_models=[model_path],
        enable_speex_noise_suppression=False,
        vad_threshold=0.0)
    outputs = []
    for i in tqdm.tqdm(os.listdir(test_file_path)[start_end[0]:start_end[1]]):
        scores = oww.predict_clip(os.path.join(test_file_path,i))
        outputs.append(np.max([j[model_name] for j in scores]))

    recall = len(np.where(np.array(outputs)>threshold)[0])/len(outputs)
    return recall

def increase_last_decimal(x):
    s = str(x)
    if '.' in s:
        decimals = len(s.split('.')[-1])
        return round(x + 10 ** (-decimals), decimals)
    else:
        return x + 1
    
import math

def ceil_last_decimal(x):
    s = str(x)
    if '.' in s:
        decimals = len(s.split('.')[-1])
        step = 10 ** (-decimals)
        # Làm tròn lên tới bậc thập phân cuối cùng
        return round(math.ceil(x / step) * step, decimals)
    else:
        return math.ceil(x)
    
def test_false_positive(model_path, model_name, test_file_paths, start_end=[0,500]):
    oww = openwakeword.Model(
        wakeword_models=[model_path],
        enable_speex_noise_suppression=False,
        vad_threshold=0.0
    )
    outputs = []
    fpph = 0
    fn = 0
    all = 0
    durations = 0

    results = []
    for test_file_path in test_file_paths:
        for file in tqdm.tqdm(os.listdir(test_file_path)[start_end[0]:start_end[1]]):
            if file.endswith(".wav"):
                scores = oww.predict_clip(os.path.join(test_file_path,file))
                filepath = os.path.join(test_file_path, file)
                with contextlib.closing(wave.open(filepath, 'r')) as f:
                    frames = f.getnframes()
                    rate = f.getframerate()
                    duration = frames / float(rate) / 3600
                results+=[s[model_name] for s in scores]
                durations+=duration
    
    excepted_fpph = [0,1,2,3,4,5,10]
    topn = sorted(results, reverse=True)
    print(f"Model: {model_name}")
    print(f"Number of test cases: {all}")
    print(f"Max wrong score: {np.max(results)}")
    print(f"Durations: {durations}")

    outputs = []
    for e,i in enumerate(excepted_fpph):
        threshold = topn[int(i*durations)]
        fp=len(np.where(np.array(results)>threshold)[0])
        fpph=fp/durations
        if threshold!=1:
            r1=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_1_thu_am_tu_dien_thoai_trung_tam_cong_nghe\converted_wav",threshold=threshold)
            r2=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\\",threshold=threshold)
            r3=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos_2\\",threshold=threshold)
            r4=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\\",threshold=threshold)
            r5=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test\\",threshold=threshold)
            r6=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test_hard\\",threshold=threshold)
        else:
            r1,r2,r3,r4,r5,r6 = 0,0,0,0,0,0
        print(f"With requirement: False Positive per hour = {i} - excepted FP = {int(i*durations)}")
        print(f"Top wrong score: {topn[i]} - Threshold: {threshold} - FP: {fp} - FPPH(box): {fpph} - Recall(phone): {r1} - Recall(box): {r2} - Recall(box_2): {r3} - Recall(box_teamtest_silent_20251117): {r4} - Recall(box_teamtest_noise_20251118): {r5} - Recall(box_teamtest_noise_20251118_hard): {r6}")
        outputs.append([e,model_name,i,threshold,fpph,r1,r2,r3,r4,r5,r6]) 
    df = pd.DataFrame(outputs,columns=['STT','Model','Expected FPPH','Threshold','FPPH','Recall (phone)','Recall (box)','Recall (box_2)','Recall (box_teamtest_silent_20251117)','Recall (box_teamtest_noise_20251118)','Recall (box_teamtest_noise_20251118_hard)'])
    df.to_excel(r'C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_checkpoints_doc\\'+f'{model_name}.xlsx',index=False)

def test_wrong_answer(model_path, model_name, test_file_path, start_end=[0,10000], threshold=0.2):
    oww = openwakeword.Model(
        wakeword_models=[model_path],
        enable_speex_noise_suppression=False,
        vad_threshold=0.0)
    outputs = []
    for i in tqdm.tqdm(os.listdir(test_file_path)[start_end[0]:start_end[1]]):
        scores = oww.predict_clip(os.path.join(test_file_path,i))
        score = np.max([j[model_name] for j in scores])
        if score<threshold:
            outputs.append(i)
    return outputs

def get_wrong_answer(model_path, model_name, threshold):
    r1=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_1_thu_am_tu_dien_thoai_trung_tam_cong_nghe\converted_wav",threshold=threshold)
    r2=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\\",threshold=threshold)
    r3=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos_2\\",threshold=threshold)
    r4=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\\",threshold=threshold)
    r5=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test\\",threshold=threshold)
    r6=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test_hard\\",threshold=threshold)

    return [r1,r2,r3,r4,r5,r6]

In [8]:
test_false_positive("../openWakeWord/model/checkpoints_cnn/old/my_model_p2_11_v4.onnx",'my_model_p2_11_v4'
                    ,[r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:16<00:00,  2.43it/s]


Model: my_model_p2_11_v4
Number of test cases: 0
Max wrong score: 0.9405883550643921
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:02<00:00,  6.02it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9405883550643921 - Threshold: 0.9405883550643921 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.45977011494252873 - Recall(box): 0.43410852713178294 - Recall(box_2): 0.43478260869565216 - Recall(box_teamtest_silent_20251117): 0.15272244355909695 - Recall(box_teamtest_noise_20251118): 0.024291497975708502 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.06it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.033563971519470215 - Threshold: 0.02968233823776245 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.5402298850574713 - Recall(box): 0.5426356589147286 - Recall(box_2): 0.4782608695652174 - Recall(box_teamtest_silent_20251117): 0.22177954847277556 - Recall(box_teamtest_noise_20251118): 0.08906882591093117 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.65it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.02968233823776245 - Threshold: 0.028688818216323853 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.5402298850574713 - Recall(box): 0.5426356589147286 - Recall(box_2): 0.4782608695652174 - Recall(box_teamtest_silent_20251117): 0.22177954847277556 - Recall(box_teamtest_noise_20251118): 0.08906882591093117 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.46it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.012973546981811523 - Threshold: 0.007957249879837036 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.5977011494252874 - Recall(box): 0.6046511627906976 - Recall(box_2): 0.5478260869565217 - Recall(box_teamtest_silent_20251117): 0.26162018592297476 - Recall(box_teamtest_noise_20251118): 0.11740890688259109 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [2]:
87+129+115+753+247

1331

In [9]:
test_false_positive("../openWakeWord/model/checkpoints_cnn/old/tammi_new_model2.onnx",'tammi_new_model2'
                    ,[r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:31<00:00,  1.22it/s]


Model: tammi_new_model2
Number of test cases: 0
Max wrong score: 0.9997017979621887
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.14it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9997017979621887 - Threshold: 0.9997017979621887 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.42528735632183906 - Recall(box): 0.24031007751937986 - Recall(box_2): 0.25217391304347825 - Recall(box_teamtest_silent_20251117): 0.1301460823373174 - Recall(box_teamtest_noise_20251118): 0.024291497975708502 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.45it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9982582330703735 - Threshold: 0.9977986812591553 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.5517241379310345 - Recall(box): 0.27906976744186046 - Recall(box_2): 0.3391304347826087 - Recall(box_teamtest_silent_20251117): 0.20053120849933598 - Recall(box_teamtest_noise_20251118): 0.05263157894736842 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.64it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9977986812591553 - Threshold: 0.9927467107772827 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.5862068965517241 - Recall(box): 0.3333333333333333 - Recall(box_2): 0.3739130434782609 - Recall(box_teamtest_silent_20251117): 0.24169986719787517 - Recall(box_teamtest_noise_20251118): 0.07692307692307693 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.25it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9727275371551514 - Threshold: 0.8535876274108887 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.6436781609195402 - Recall(box): 0.46511627906976744 - Recall(box_2): 0.48695652173913045 - Recall(box_teamtest_silent_20251117): 0.3094289508632138 - Recall(box_teamtest_noise_20251118): 0.145748987854251 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [10]:
test_false_positive("model/old/tammi_oi_20251016_17_20/tammi_oi_20251016_17_20.onnx",'tammi_oi_20251016_17_20',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:54<00:00,  1.13it/s]


Model: tammi_oi_20251016_17_20
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.63it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.28it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999998807907104 - Threshold: 0.9999997615814209 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.8735632183908046 - Recall(box): 0.6046511627906976 - Recall(box_2): 0.7391304347826086 - Recall(box_teamtest_silent_20251117): 0.2908366533864542 - Recall(box_teamtest_noise_20251118): 0.1862348178137652 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.43it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999997615814209 - Threshold: 0.9999971389770508 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.896551724137931 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.33731739707835323 - Recall(box_teamtest_noise_20251118): 0.22267206477732793 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.22it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999945163726807 - Threshold: 0.9999737739562988 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7054263565891473 - Recall(box_2): 0.808695652173913 - Recall(box_teamtest_silent_20251117): 0.37317397078353254 - Recall(box_teamtest_noise_20251118): 0.2550607287449393 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [11]:
test_false_positive("model/old/tammi_oi_20251016_17_20_22/tammi_oi_20251016_17_20_22.onnx",'tammi_oi_20251016_17_20_22',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:44<00:00,  1.17it/s]


Model: tammi_oi_20251016_17_20_22
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.37it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.74it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999998807907104 - Threshold: 0.9999998211860657 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8347826086956521 - Recall(box_teamtest_silent_20251117): 0.46215139442231074 - Recall(box_teamtest_noise_20251118): 0.31983805668016196 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.57it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999998211860657 - Threshold: 0.9999997615814209 - FP: 6 - FPPH(box): 4.274020620418115 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8434782608695652 - Recall(box_teamtest_silent_20251117): 0.4648074369189907 - Recall(box_teamtest_noise_20251118): 0.31983805668016196 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.54it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999995827674866 - Threshold: 0.9999986886978149 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7286821705426356 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.5258964143426295 - Recall(box_teamtest_noise_20251118): 0.3724696356275304 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [12]:
test_false_positive("model/old/tammi_oi_20251016_20251017/tammi_oi_20251016_20251017.onnx",'tammi_oi_20251016_20251017',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:32<00:00,  1.22it/s]


Model: tammi_oi_20251016_20251017
Number of test cases: 0
Max wrong score: 0.9999997019767761
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.62it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999997019767761 - Threshold: 0.9999997019767761 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.7931034482758621 - Recall(box): 0.5271317829457365 - Recall(box_2): 0.6434782608695652 - Recall(box_teamtest_silent_20251117): 0.20318725099601595 - Recall(box_teamtest_noise_20251118): 0.11740890688259109 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.72it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999774098396301 - Threshold: 0.9999624490737915 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.8850574712643678 - Recall(box): 0.627906976744186 - Recall(box_2): 0.8173913043478261 - Recall(box_teamtest_silent_20251117): 0.33731739707835323 - Recall(box_teamtest_noise_20251118): 0.22267206477732793 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999624490737915 - Threshold: 0.9999527335166931 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.8850574712643678 - Recall(box): 0.627906976744186 - Recall(box_2): 0.8173913043478261 - Recall(box_teamtest_silent_20251117): 0.3386454183266932 - Recall(box_teamtest_noise_20251118): 0.23076923076923078 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00,  9.87it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9998877048492432 - Threshold: 0.9997260570526123 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6666666666666666 - Recall(box_2): 0.8347826086956521 - Recall(box_teamtest_silent_20251117): 0.3798140770252324 - Recall(box_teamtest_noise_20251118): 0.2550607287449393 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [13]:
test_false_positive("model/old/tammi_oi_20251016/tammi_oi_20251016.onnx",'tammi_oi_20251016',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:31<00:00,  2.19it/s]


Model: tammi_oi_20251016
Number of test cases: 0
Max wrong score: 0.9999997615814209
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.09it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999997615814209 - Threshold: 0.9999997615814209 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8160919540229885 - Recall(box): 0.5348837209302325 - Recall(box_2): 0.591304347826087 - Recall(box_teamtest_silent_20251117): 0.20318725099601595 - Recall(box_teamtest_noise_20251118): 0.11740890688259109 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999990463256836 - Threshold: 0.9999982118606567 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.8505747126436781 - Recall(box): 0.5736434108527132 - Recall(box_2): 0.6782608695652174 - Recall(box_teamtest_silent_20251117): 0.25763612217795484 - Recall(box_teamtest_noise_20251118): 0.15789473684210525 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.96it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999982118606567 - Threshold: 0.9999783039093018 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.896551724137931 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.7391304347826086 - Recall(box_teamtest_silent_20251117): 0.3147410358565737 - Recall(box_teamtest_noise_20251118): 0.22267206477732793 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.25it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9998741149902344 - Threshold: 0.9997798204421997 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.7652173913043478 - Recall(box_teamtest_silent_20251117): 0.35856573705179284 - Recall(box_teamtest_noise_20251118): 0.26720647773279355 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [14]:
test_false_positive("model/old/tammi_oi_20251016_17_20_22/tammi_oi_20251016_17_20_22_150ks.onnx",'tammi_oi_20251016_17_20_22_150ks',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:41<00:00,  2.06it/s]


Model: tammi_oi_20251016_17_20_22_150ks
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.40it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.35it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999997615814209 - Threshold: 0.9999997615814209 - FP: 3 - FPPH(box): 2.1370103102090576 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.8521739130434782 - Recall(box_teamtest_silent_20251117): 0.5033200531208499 - Recall(box_teamtest_noise_20251118): 0.3319838056680162 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.67it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999997615814209 - Threshold: 0.9999990463256836 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.5405046480743692 - Recall(box_teamtest_noise_20251118): 0.3724696356275304 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.95it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999985694885254 - Threshold: 0.9999976754188538 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7674418604651163 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.5617529880478087 - Recall(box_teamtest_noise_20251118): 0.38461538461538464 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [15]:
test_false_positive("model/old/tammi_oi_20251016_17_20_22/tammi_oi_20251016_17_20_22_100ks_50ks.onnx",'tammi_oi_20251016_17_20_22_100ks_50ks',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:12<00:00,  1.72it/s]


Model: tammi_oi_20251016_17_20_22_100ks_50ks
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.59it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:04<00:00,  4.19it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 1.0 - Threshold: 0.9999999403953552 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8260869565217391 - Recall(box_teamtest_silent_20251117): 0.4754316069057105 - Recall(box_teamtest_noise_20251118): 0.32388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.66it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999999403953552 - Threshold: 0.9999998807907104 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9080459770114943 - Recall(box): 0.689922480620155 - Recall(box_2): 0.8347826086956521 - Recall(box_teamtest_silent_20251117): 0.48738379814077026 - Recall(box_teamtest_noise_20251118): 0.32793522267206476 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999998807907104 - Threshold: 0.9999997615814209 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.8521739130434782 - Recall(box_teamtest_silent_20251117): 0.5166002656042497 - Recall(box_teamtest_noise_20251118): 0.3603238866396761 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [16]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23/tammi_oi_20251016_17_20_23.onnx",'tammi_oi_20251016_17_20_23',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:22<00:00,  2.33it/s]


Model: tammi_oi_20251016_17_20_23
Number of test cases: 0
Max wrong score: 0.9999988079071045
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.60it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999988079071045 - Threshold: 0.9999988079071045 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6589147286821705 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.3612217795484728 - Recall(box_teamtest_noise_20251118): 0.2145748987854251 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.53it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9998917579650879 - Threshold: 0.9998599290847778 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.45683930942895085 - Recall(box_teamtest_noise_20251118): 0.3076923076923077 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9998599290847778 - Threshold: 0.9993094801902771 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7751937984496124 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.4807436918990704 - Recall(box_teamtest_noise_20251118): 0.3319838056680162 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.37it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9971132278442383 - Threshold: 0.995839536190033 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9310344827586207 - Recall(box): 0.8062015503875969 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.5126162018592297 - Recall(box_teamtest_noise_20251118): 0.3643724696356275 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [17]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23/tammi_oi_20251016_17_20_23_pt.onnx",'tammi_oi_20251016_17_20_23_pt',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:21<00:00,  2.35it/s]


Model: tammi_oi_20251016_17_20_23_pt
Number of test cases: 0
Max wrong score: 0.9999997615814209
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999997615814209 - Threshold: 0.9999997615814209 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.37715803452855245 - Recall(box_teamtest_noise_20251118): 0.23076923076923078 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.40it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999414682388306 - Threshold: 0.999939501285553 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7596899224806202 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.48738379814077026 - Recall(box_teamtest_noise_20251118): 0.32388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.41it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.999939501285553 - Threshold: 0.9998652338981628 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7751937984496124 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.49800796812749004 - Recall(box_teamtest_noise_20251118): 0.3360323886639676 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9996771812438965 - Threshold: 0.9995628595352173 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7906976744186046 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.5152722443559097 - Recall(box_teamtest_noise_20251118): 0.3603238866396761 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [3]:
test_false_positive("model/old/tammi_oi_20251016_17_20_31/tammi_oi_20251016_17_20_31.onnx",'tammi_oi_20251016_17_20_31',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:14<00:00,  2.46it/s]


Model: tammi_oi_20251016_17_20_31
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:02<00:00,  6.00it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.60it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999996423721313 - Threshold: 0.9999995231628418 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9310344827586207 - Recall(box): 0.7286821705426356 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.37317397078353254 - Recall(box_teamtest_noise_20251118): 0.25101214574898784 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:02<00:00,  6.11it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999995231628418 - Threshold: 0.9999991655349731 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9425287356321839 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.38247011952191234 - Recall(box_teamtest_noise_20251118): 0.2550607287449393 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.33it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.999996542930603 - Threshold: 0.9999876022338867 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9425287356321839 - Recall(box): 0.7906976744186046 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.4103585657370518 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [4]:
test_false_positive("model/old/tammi_oi_20251031/tammi_oi_20251031.onnx",'tammi_oi_20251031',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:37<00:00,  1.20it/s]


Model: tammi_oi_20251031
Number of test cases: 0
Max wrong score: 0.999999463558197
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.84it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.999999463558197 - Threshold: 0.999999463558197 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.627906976744186 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.3930942895086321 - Recall(box_teamtest_noise_20251118): 0.26720647773279355 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.74it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.999988317489624 - Threshold: 0.9999833703041077 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.47277556440903057 - Recall(box_teamtest_noise_20251118): 0.3076923076923077 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.76it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999833703041077 - Threshold: 0.9999781847000122 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.47941567065073043 - Recall(box_teamtest_noise_20251118): 0.3076923076923077 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.71it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999688863754272 - Threshold: 0.9999213218688965 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.7674418604651163 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.4953519256308101 - Recall(box_teamtest_noise_20251118): 0.32793522267206476 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [5]:
test_false_positive("model/old/tammi_oi_20251103_aug_1/tammi_oi_20251103_aug_1.onnx",'tammi_oi_20251103_aug_1',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:50<00:00,  1.44it/s]


Model: tammi_oi_20251103_aug_1
Number of test cases: 0
Max wrong score: 0.9991925954818726
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.53it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9991925954818726 - Threshold: 0.9991925954818726 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8850574712643678 - Recall(box): 0.5968992248062015 - Recall(box_2): 0.6956521739130435 - Recall(box_teamtest_silent_20251117): 0.250996015936255 - Recall(box_teamtest_noise_20251118): 0.16194331983805668 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.15it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9564812183380127 - Threshold: 0.9539432525634766 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.627906976744186 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.34130146082337315 - Recall(box_teamtest_noise_20251118): 0.242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:04<00:00,  3.99it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9539432525634766 - Threshold: 0.9262714982032776 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.3466135458167331 - Recall(box_teamtest_noise_20251118): 0.2550607287449393 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.29it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.8836001753807068 - Threshold: 0.7074055075645447 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8260869565217391 - Recall(box_teamtest_silent_20251117): 0.38114209827357237 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [6]:
test_false_positive("model/old/tammi_oi_20251103_aug_2/tammi_oi_20251103_aug_2.onnx",'tammi_oi_20251103_aug_2',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:44<00:00,  1.17it/s]


Model: tammi_oi_20251103_aug_2
Number of test cases: 0
Max wrong score: 0.9999927282333374
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.16it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999927282333374 - Threshold: 0.9999927282333374 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8045977011494253 - Recall(box): 0.4728682170542636 - Recall(box_2): 0.5304347826086957 - Recall(box_teamtest_silent_20251117): 0.16069057104913678 - Recall(box_teamtest_noise_20251118): 0.08502024291497975 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.72it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9993904829025269 - Threshold: 0.9991952180862427 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.896551724137931 - Recall(box): 0.5426356589147286 - Recall(box_2): 0.6956521739130435 - Recall(box_teamtest_silent_20251117): 0.26693227091633465 - Recall(box_teamtest_noise_20251118): 0.15789473684210525 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.87it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9991952180862427 - Threshold: 0.996566116809845 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.896551724137931 - Recall(box): 0.5891472868217055 - Recall(box_2): 0.7304347826086957 - Recall(box_teamtest_silent_20251117): 0.30677290836653387 - Recall(box_teamtest_noise_20251118): 0.1862348178137652 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:04<00:00,  3.76it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9958882331848145 - Threshold: 0.986579179763794 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6124031007751938 - Recall(box_2): 0.7478260869565218 - Recall(box_teamtest_silent_20251117): 0.3346613545816733 - Recall(box_teamtest_noise_20251118): 0.20242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [7]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23_20251103_raw/tammi_oi_20251016_17_20_23_20251103_raw.onnx",'tammi_oi_20251016_17_20_23_20251103_raw',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:47<00:00,  1.15it/s]


Model: tammi_oi_20251016_17_20_23_20251103_raw
Number of test cases: 0
Max wrong score: 0.5898623466491699
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.43it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.5898623466491699 - Threshold: 0.5898623466491699 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5891472868217055 - Recall(box_2): 0.7043478260869566 - Recall(box_teamtest_silent_20251117): 0.2204515272244356 - Recall(box_teamtest_noise_20251118): 0.10931174089068826 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.54it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.20745575428009033 - Threshold: 0.14950919151306152 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6589147286821705 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.2735723771580345 - Recall(box_teamtest_noise_20251118): 0.16194331983805668 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.14950919151306152 - Threshold: 0.12996366620063782 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6666666666666666 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.28021248339973437 - Recall(box_teamtest_noise_20251118): 0.1700404858299595 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.13it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.09402269124984741 - Threshold: 0.08588993549346924 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9540229885057471 - Recall(box): 0.689922480620155 - Recall(box_2): 0.808695652173913 - Recall(box_teamtest_silent_20251117): 0.28818061088977426 - Recall(box_teamtest_noise_20251118): 0.18218623481781376 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [3]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23_20251103_raw/tammi_oi_20251016_17_20_23_20251103_raw_2.onnx",'tammi_oi_20251016_17_20_23_20251103_raw_2',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:05<00:00,  1.35it/s]  


Model: tammi_oi_20251016_17_20_23_20251103_raw_2
Number of test cases: 0
Max wrong score: 0.7721020579338074
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.48it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.7721020579338074 - Threshold: 0.7721020579338074 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.7652173913043478 - Recall(box_teamtest_silent_20251117): 0.2589641434262948 - Recall(box_teamtest_noise_20251118): 0.1659919028340081 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.75it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.16922533512115479 - Threshold: 0.12239906191825867 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7364341085271318 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.33731739707835323 - Recall(box_teamtest_noise_20251118): 0.23481781376518218 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.11it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.12239906191825867 - Threshold: 0.09164834022521973 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.751937984496124 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.3452855245683931 - Recall(box_teamtest_noise_20251118): 0.2388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.30it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.033045053482055664 - Threshold: 0.020477741956710815 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7906976744186046 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.4169986719787517 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [8]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23_20251103_raw/tammi_oi_20251016_17_20_23_20251103_raw_3.onnx",'tammi_oi_20251016_17_20_23_20251103_raw_3',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:21<00:00,  2.34it/s]


Model: tammi_oi_20251016_17_20_23_20251103_raw_3
Number of test cases: 0
Max wrong score: 0.9976397752761841
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9976397752761841 - Threshold: 0.9976397752761841 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8850574712643678 - Recall(box): 0.5736434108527132 - Recall(box_2): 0.6434782608695652 - Recall(box_teamtest_silent_20251117): 0.21646746347941567 - Recall(box_teamtest_noise_20251118): 0.11336032388663968 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.37it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.631573498249054 - Threshold: 0.5159198045730591 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9425287356321839 - Recall(box): 0.689922480620155 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.3200531208499336 - Recall(box_teamtest_noise_20251118): 0.19433198380566802 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.38it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.5159198045730591 - Threshold: 0.23188334703445435 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9540229885057471 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.35856573705179284 - Recall(box_teamtest_noise_20251118): 0.22672064777327935 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.54it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.1862291693687439 - Threshold: 0.07952791452407837 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9885057471264368 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.4103585657370518 - Recall(box_teamtest_noise_20251118): 0.2834008097165992 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [9]:
test_false_positive("model/old/augment_data_2025101728_20251105/augment_data_2025101728_20251105.onnx",'augment_data_2025101728_20251105',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:55<00:00,  1.12it/s]


Model: augment_data_2025101728_20251105
Number of test cases: 0
Max wrong score: 0.999756395816803
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.37it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.999756395816803 - Threshold: 0.999756395816803 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8735632183908046 - Recall(box): 0.5038759689922481 - Recall(box_2): 0.6434782608695652 - Recall(box_teamtest_silent_20251117): 0.25232403718459495 - Recall(box_teamtest_noise_20251118): 0.12955465587044535 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.51it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.996462881565094 - Threshold: 0.9925820231437683 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.7304347826086957 - Recall(box_teamtest_silent_20251117): 0.36387782204515273 - Recall(box_teamtest_noise_20251118): 0.23481781376518218 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.36it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9925820231437683 - Threshold: 0.9859530925750732 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6511627906976745 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.38114209827357237 - Recall(box_teamtest_noise_20251118): 0.24696356275303644 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.40it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9656862020492554 - Threshold: 0.9505224823951721 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.808695652173913 - Recall(box_teamtest_silent_20251117): 0.4196547144754316 - Recall(box_teamtest_noise_20251118): 0.2834008097165992 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [10]:
test_false_positive("model/old/augment_data_2025101728_20251105/augment_data_2025101728_20251105_100k.onnx",'augment_data_2025101728_20251105_100k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:53<00:00,  1.13it/s]


Model: augment_data_2025101728_20251105_100k
Number of test cases: 0
Max wrong score: 0.9999961853027344
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  4.54it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999961853027344 - Threshold: 0.9999961853027344 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.49612403100775193 - Recall(box_2): 0.5652173913043478 - Recall(box_teamtest_silent_20251117): 0.25630810092961487 - Recall(box_teamtest_noise_20251118): 0.1417004048582996 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.27it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999451637268066 - Threshold: 0.9999417066574097 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5581395348837209 - Recall(box_2): 0.6695652173913044 - Recall(box_teamtest_silent_20251117): 0.30677290836653387 - Recall(box_teamtest_noise_20251118): 0.1862348178137652 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.71it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999417066574097 - Threshold: 0.9998854398727417 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.5813953488372093 - Recall(box_2): 0.6956521739130435 - Recall(box_teamtest_silent_20251117): 0.3293492695883134 - Recall(box_teamtest_noise_20251118): 0.19433198380566802 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.56it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9998582005500793 - Threshold: 0.9997595548629761 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6046511627906976 - Recall(box_2): 0.7130434782608696 - Recall(box_teamtest_silent_20251117): 0.34926958831341304 - Recall(box_teamtest_noise_20251118): 0.20647773279352227 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [11]:
test_false_positive("model/old/tammi_oi_20251016_17_20_23_20251103_raw/tammi_oi_20251016_17_20_23_20251103_raw_2000k.onnx",'tammi_oi_20251016_17_20_23_20251103_raw_2000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:57<00:00,  1.12it/s]


Model: tammi_oi_20251016_17_20_23_20251103_raw_2000k
Number of test cases: 0
Max wrong score: 0.9968402981758118
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.12it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9968402981758118 - Threshold: 0.9968402981758118 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.7391304347826086 - Recall(box_teamtest_silent_20251117): 0.2855245683930943 - Recall(box_teamtest_noise_20251118): 0.16194331983805668 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.78it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.324306845664978 - Threshold: 0.2970731854438782 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9885057471264368 - Recall(box): 0.7364341085271318 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.43293492695883135 - Recall(box_teamtest_noise_20251118): 0.31983805668016196 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.33it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.2970731854438782 - Threshold: 0.20210230350494385 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9885057471264368 - Recall(box): 0.751937984496124 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.45285524568393093 - Recall(box_teamtest_noise_20251118): 0.3441295546558704 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.57it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.16201704740524292 - Threshold: 0.13356348872184753 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9885057471264368 - Recall(box): 0.7596899224806202 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.47277556440903057 - Recall(box_teamtest_noise_20251118): 0.3481781376518219 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [12]:
test_false_positive("model/old/tammi_oi_20251110/tammi_oi_20251110.onnx",'tammi_oi_20251110',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:47<00:00,  1.15it/s]


Model: tammi_oi_20251110
Number of test cases: 0
Max wrong score: 0.9968008995056152
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  4.62it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9968008995056152 - Threshold: 0.9968008995056152 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5813953488372093 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.36254980079681276 - Recall(box_teamtest_noise_20251118): 0.25101214574898784 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.38it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.7518974542617798 - Threshold: 0.47582149505615234 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.689922480620155 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.5351925630810093 - Recall(box_teamtest_noise_20251118): 0.3805668016194332 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.47582149505615234 - Threshold: 0.36848199367523193 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.5524568393094289 - Recall(box_teamtest_noise_20251118): 0.38866396761133604 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.2857438921928406 - Threshold: 0.12370768189430237 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.9391304347826087 - Recall(box_teamtest_silent_20251117): 0.6055776892430279 - Recall(box_teamtest_noise_20251118): 0.4534412955465587 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [13]:
test_false_positive("model/old/tammi_oi_20251110/tammi_oi_20251110_800ks.onnx",'tammi_oi_20251110_800ks',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:20<00:00,  2.36it/s]


Model: tammi_oi_20251110_800ks
Number of test cases: 0
Max wrong score: 0.9710699319839478
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9710699319839478 - Threshold: 0.9710699319839478 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5736434108527132 - Recall(box_2): 0.7043478260869566 - Recall(box_teamtest_silent_20251117): 0.34926958831341304 - Recall(box_teamtest_noise_20251118): 0.20647773279352227 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.77it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.28157997131347656 - Threshold: 0.23668837547302246 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.5112881806108898 - Recall(box_teamtest_noise_20251118): 0.3724696356275304 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.48it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.23668837547302246 - Threshold: 0.11091634631156921 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.5391766268260292 - Recall(box_teamtest_noise_20251118): 0.4251012145748988 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.60it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.059668660163879395 - Threshold: 0.03052431344985962 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7364341085271318 - Recall(box_2): 0.9391304347826087 - Recall(box_teamtest_silent_20251117): 0.6122177954847278 - Recall(box_teamtest_noise_20251118): 0.46558704453441296 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [15]:
test_false_positive("model/old/tammi_oi_20251111/tammi_oi_20251111.onnx",'tammi_oi_20251111',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:06<00:00,  1.78it/s]


Model: tammi_oi_20251111
Number of test cases: 0
Max wrong score: 0.9998306035995483
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.35it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9998306035995483 - Threshold: 0.9998306035995483 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8735632183908046 - Recall(box): 0.5038759689922481 - Recall(box_2): 0.6521739130434783 - Recall(box_teamtest_silent_20251117): 0.23638778220451528 - Recall(box_teamtest_noise_20251118): 0.1417004048582996 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:04<00:00,  4.09it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.7246831655502319 - Threshold: 0.5497028827667236 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.689922480620155 - Recall(box_2): 0.8434782608695652 - Recall(box_teamtest_silent_20251117): 0.42363877822045154 - Recall(box_teamtest_noise_20251118): 0.29959514170040485 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:04<00:00,  4.41it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.5497028827667236 - Threshold: 0.5381301045417786 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.689922480620155 - Recall(box_2): 0.8434782608695652 - Recall(box_teamtest_silent_20251117): 0.42895086321381143 - Recall(box_teamtest_noise_20251118): 0.30364372469635625 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.29it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.24166062474250793 - Threshold: 0.1360567808151245 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.751937984496124 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.4940239043824701 - Recall(box_teamtest_noise_20251118): 0.3724696356275304 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [16]:
test_false_positive("model/tammi_oi_20251112/tammi_oi_20251112.onnx",'tammi_oi_20251112',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [05:05<00:00,  1.09it/s]


Model: tammi_oi_20251112
Number of test cases: 0
Max wrong score: 0.9171172380447388
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.01it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9171172380447388 - Threshold: 0.9171172380447388 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9310344827586207 - Recall(box): 0.5968992248062015 - Recall(box_2): 0.7652173913043478 - Recall(box_teamtest_silent_20251117): 0.3253652058432935 - Recall(box_teamtest_noise_20251118): 0.1902834008097166 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.99it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.23340442776679993 - Threshold: 0.2326730191707611 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6666666666666666 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.42762284196547146 - Recall(box_teamtest_noise_20251118): 0.27125506072874495 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.65it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.2326730191707611 - Threshold: 0.18632611632347107 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.4342629482071713 - Recall(box_teamtest_noise_20251118): 0.2793522267206478 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.92it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.11058613657951355 - Threshold: 0.053249359130859375 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.49667994687915007 - Recall(box_teamtest_noise_20251118): 0.3319838056680162 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [17]:
test_false_positive("model/tammi_oi_20251113/tammi_oi_20251113.onnx",'tammi_oi_20251113',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:04<00:00,  1.80it/s]


Model: tammi_oi_20251113
Number of test cases: 0
Max wrong score: 0.9999996423721313
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.91it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999996423721313 - Threshold: 0.9999996423721313 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.5348837209302325 - Recall(box_2): 0.6521739130434783 - Recall(box_teamtest_silent_20251117): 0.2403718459495352 - Recall(box_teamtest_noise_20251118): 0.145748987854251 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.97it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9981998205184937 - Threshold: 0.9962488412857056 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6124031007751938 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.33598937583001326 - Recall(box_teamtest_noise_20251118): 0.20242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.78it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9962488412857056 - Threshold: 0.9828939437866211 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.627906976744186 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.351925630810093 - Recall(box_teamtest_noise_20251118): 0.21052631578947367 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.97it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9373859167098999 - Threshold: 0.7013831734657288 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6744186046511628 - Recall(box_2): 0.8521739130434782 - Recall(box_teamtest_silent_20251117): 0.4077025232403719 - Recall(box_teamtest_noise_20251118): 0.242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [18]:
test_false_positive("model/tammi_oi_20251113_2/tammi_oi_20251113_2.onnx",'tammi_oi_20251113_2',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:20<00:00,  2.36it/s]


Model: tammi_oi_20251113_2
Number of test cases: 0
Max wrong score: 0.9900082349777222
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:12<00:00,  1.43it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9900082349777222 - Threshold: 0.9900082349777222 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9310344827586207 - Recall(box): 0.49612403100775193 - Recall(box_2): 0.6521739130434783 - Recall(box_teamtest_silent_20251117): 0.2908366533864542 - Recall(box_teamtest_noise_20251118): 0.1700404858299595 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:08<00:00,  2.11it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.8373374342918396 - Threshold: 0.6556577682495117 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6124031007751938 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.3784860557768924 - Recall(box_teamtest_noise_20251118): 0.22672064777327935 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:10<00:00,  1.76it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.6556577682495117 - Threshold: 0.3182862401008606 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.4209827357237716 - Recall(box_teamtest_noise_20251118): 0.25101214574898784 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:11<00:00,  1.53it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.141584575176239 - Threshold: 0.07629156112670898 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.689922480620155 - Recall(box_2): 0.8434782608695652 - Recall(box_teamtest_silent_20251117): 0.47808764940239046 - Recall(box_teamtest_noise_20251118): 0.3117408906882591 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [26]:
test_false_positive("model/tammi_oi_20251114/tammi_oi_20251114.onnx",'tammi_oi_20251114',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:22<00:00,  2.33it/s]


Model: tammi_oi_20251114
Number of test cases: 0
Max wrong score: 0.9904156923294067
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.64it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9904156923294067 - Threshold: 0.9904156923294067 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.3837981407702523 - Recall(box_teamtest_noise_20251118): 0.23481781376518218 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.71it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.7663524746894836 - Threshold: 0.49695080518722534 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.8869565217391304 - Recall(box_teamtest_silent_20251117): 0.46613545816733065 - Recall(box_teamtest_noise_20251118): 0.29959514170040485 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.61it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.49695080518722534 - Threshold: 0.4317131042480469 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.4767596281540505 - Recall(box_teamtest_noise_20251118): 0.3076923076923077 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.67it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.3785606622695923 - Threshold: 0.21288523077964783 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7286821705426356 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.5086321381142098 - Recall(box_teamtest_noise_20251118): 0.340080971659919 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [25]:
test_false_positive("model/tammi_oi_20251120/tammi_oi_20251120.onnx",'tammi_oi_20251120',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:21<00:00,  2.34it/s]


Model: tammi_oi_20251120
Number of test cases: 0
Max wrong score: 0.9374750852584839
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.71it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9374750852584839 - Threshold: 0.9374750852584839 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.6046511627906976 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.3798140770252324 - Recall(box_teamtest_noise_20251118): 0.242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.74it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.7416525483131409 - Threshold: 0.4624183475971222 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.4594953519256308 - Recall(box_teamtest_noise_20251118): 0.29554655870445345 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.78it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.4624183475971222 - Threshold: 0.3561546206474304 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.47410358565737054 - Recall(box_teamtest_noise_20251118): 0.3117408906882591 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.74it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.14335468411445618 - Threshold: 0.03251039981842041 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.751937984496124 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.5617529880478087 - Recall(box_teamtest_noise_20251118): 0.39271255060728744 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [24]:
test_false_positive("model/tammi_oi_20251120/tammi_oi_20251120_2000k.onnx",'tammi_oi_20251120_2000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:27<00:00,  2.26it/s]


Model: tammi_oi_20251120_2000k
Number of test cases: 0
Max wrong score: 0.9997915029525757
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.55it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9997915029525757 - Threshold: 0.9997915029525757 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.5426356589147286 - Recall(box_2): 0.7217391304347827 - Recall(box_teamtest_silent_20251117): 0.3851261620185923 - Recall(box_teamtest_noise_20251118): 0.24696356275303644 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.62it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9963597059249878 - Threshold: 0.9934925436973572 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9310344827586207 - Recall(box): 0.5968992248062015 - Recall(box_2): 0.808695652173913 - Recall(box_teamtest_silent_20251117): 0.45152722443559096 - Recall(box_teamtest_noise_20251118): 0.27530364372469635 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.68it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9934925436973572 - Threshold: 0.9520837068557739 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9540229885057471 - Recall(box): 0.6201550387596899 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.4940239043824701 - Recall(box_teamtest_noise_20251118): 0.3117408906882591 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.87it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.5379453301429749 - Threshold: 0.17711448669433594 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.600265604249668 - Recall(box_teamtest_noise_20251118): 0.4251012145748988 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [23]:
test_false_positive("model/tammi_oi_20251120/tammi_oi_20251120_2000k_suacode.onnx",'tammi_oi_20251120_2000k_suacode',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [19:56<00:00,  3.60s/it]  


Model: tammi_oi_20251120_2000k_suacode
Number of test cases: 0
Max wrong score: 0.9999996423721313
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:14<00:00,  1.23it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999996423721313 - Threshold: 0.9999996423721313 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9080459770114943 - Recall(box): 0.5891472868217055 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.399734395750332 - Recall(box_teamtest_noise_20251118): 0.2631578947368421 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:10<00:00,  1.73it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9996041059494019 - Threshold: 0.9930715560913086 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.8869565217391304 - Recall(box_teamtest_silent_20251117): 0.5604249667994687 - Recall(box_teamtest_noise_20251118): 0.38461538461538464 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:13<00:00,  1.30it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9930715560913086 - Threshold: 0.9850941896438599 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8869565217391304 - Recall(box_teamtest_silent_20251117): 0.5723771580345286 - Recall(box_teamtest_noise_20251118): 0.4008097165991903 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.02it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9305441379547119 - Threshold: 0.8200501203536987 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.6175298804780877 - Recall(box_teamtest_noise_20251118): 0.44534412955465585 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [22]:
test_false_positive("model/tammi_oi_20251125/tammi_oi_20251125.onnx",'tammi_oi_20251125',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [20:04<00:00,  3.63s/it]  


Model: tammi_oi_20251125
Number of test cases: 0
Max wrong score: 0.9999717473983765
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:14<00:00,  1.25it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999717473983765 - Threshold: 0.9999717473983765 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5891472868217055 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.3253652058432935 - Recall(box_teamtest_noise_20251118): 0.21862348178137653 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:13<00:00,  1.34it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9997185468673706 - Threshold: 0.99971604347229 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.6201550387596899 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.3745019920318725 - Recall(box_teamtest_noise_20251118): 0.22672064777327935 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:15<00:00,  1.19it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.99971604347229 - Threshold: 0.9992155432701111 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.39176626826029215 - Recall(box_teamtest_noise_20251118): 0.2388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:11<00:00,  1.59it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9953745603561401 - Threshold: 0.7186934947967529 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.5325365205843293 - Recall(box_teamtest_noise_20251118): 0.3562753036437247 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [21]:
test_false_positive("model/tammi_oi_20251125/tammi_oi_20251125_phase_03_step_05264_recall_0.5327.onnx",'tammi_oi_20251125_phase_03_step_05264_recall_0.5327',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [20:06<00:00,  3.64s/it]  


Model: tammi_oi_20251125_phase_03_step_05264_recall_0.5327
Number of test cases: 0
Max wrong score: 0.9999858140945435
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:12<00:00,  1.40it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999858140945435 - Threshold: 0.9999858140945435 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5891472868217055 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.32138114209827356 - Recall(box_teamtest_noise_20251118): 0.21052631578947367 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:06<00:00,  2.88it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9998315572738647 - Threshold: 0.9998080730438232 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.627906976744186 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.37715803452855245 - Recall(box_teamtest_noise_20251118): 0.23076923076923078 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:09<00:00,  1.89it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9998080730438232 - Threshold: 0.9994834661483765 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6356589147286822 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.3930942895086321 - Recall(box_teamtest_noise_20251118): 0.2388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:12<00:00,  1.42it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9975289106369019 - Threshold: 0.8470475673675537 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8869565217391304 - Recall(box_teamtest_silent_20251117): 0.5285524568393094 - Recall(box_teamtest_noise_20251118): 0.3522267206477733 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [20]:
test_false_positive("model/tammi_oi_20251125/tammi_oi_20251125_phase_03_step_31579_recall_0.5374.onnx",'tammi_oi_20251125_phase_03_step_31579_recall_0.5374',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [18:32<00:00,  3.35s/it]  


Model: tammi_oi_20251125_phase_03_step_31579_recall_0.5374
Number of test cases: 0
Max wrong score: 0.9999963045120239
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:08<00:00,  2.20it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999963045120239 - Threshold: 0.9999963045120239 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9195402298850575 - Recall(box): 0.5813953488372093 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.3200531208499336 - Recall(box_teamtest_noise_20251118): 0.20242914979757085 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:09<00:00,  1.82it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999493360519409 - Threshold: 0.9999406337738037 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9195402298850575 - Recall(box): 0.627906976744186 - Recall(box_2): 0.782608695652174 - Recall(box_teamtest_silent_20251117): 0.3745019920318725 - Recall(box_teamtest_noise_20251118): 0.23076923076923078 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:11<00:00,  1.57it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999406337738037 - Threshold: 0.999814510345459 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.8 - Recall(box_teamtest_silent_20251117): 0.3930942895086321 - Recall(box_teamtest_noise_20251118): 0.2388663967611336 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:11<00:00,  1.61it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9985525012016296 - Threshold: 0.9487593173980713 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.5298804780876494 - Recall(box_teamtest_noise_20251118): 0.3522267206477733 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [19]:
test_false_positive("model/tammi_oi_20251125/tammi_oi_20251125_2000k.onnx",'tammi_oi_20251125_2000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [20:06<00:00,  3.63s/it]  


Model: tammi_oi_20251125_2000k
Number of test cases: 0
Max wrong score: 0.9999980926513672
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:11<00:00,  1.60it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999980926513672 - Threshold: 0.9999980926513672 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6511627906976745 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.36786188579017265 - Recall(box_teamtest_noise_20251118): 0.21052631578947367 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:14<00:00,  1.26it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999033212661743 - Threshold: 0.9998852014541626 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.8347826086956521 - Recall(box_teamtest_silent_20251117): 0.4435590969455511 - Recall(box_teamtest_noise_20251118): 0.2550607287449393 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:15<00:00,  1.18it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9998852014541626 - Threshold: 0.9987215399742126 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.4913678618857902 - Recall(box_teamtest_noise_20251118): 0.31983805668016196 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:12<00:00,  1.50it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9796262383460999 - Threshold: 0.8311673402786255 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9655172413793104 - Recall(box): 0.7751937984496124 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.598937583001328 - Recall(box_teamtest_noise_20251118): 0.4089068825910931 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [7]:
!python openwakeword/train.py --training_config openwakeword/tammi_oi.yml --export_checkpoint --checkpoint_path model/tammi_oi_20251125/phase_03_step_31579_recall_0.5374.pt --onnx_output_path model/tammi_oi_20251125/tammi_oi_20251125_phase_03_step_31579_recall_0.5374.onnx

data\test\positive_test\20251110\records\use\test_mix_001\0003F4N_R0B_250929_03_gaindown.wav
33000
✅ Exported ONNX model: model\tammi_oi_20251125\tammi_oi_20251125_phase_03_step_31579_recall_0.5374.onnx


2025-11-26 15:42:13.364742: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-26 15:42:14.669626: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
c:\Users\User\miniconda3\envs\aicamera\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see http

In [ ]:
test_false_positive("model/tammi_oi_20251126/tammi_oi_20251126.onnx",'tammi_oi_20251126',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [05:26<00:00,  1.02it/s]  


Model: tammi_oi_20251126
Number of test cases: 0
Max wrong score: 0.9999916553497314
Durations: 1.4038303819444458


100%|██████████| 117/117 [00:20<00:00,  5.77it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999916553497314 - Threshold: 0.9999916553497314 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8709677419354839 - Recall(box): 0.5970149253731343 - Recall(box_2): 0.7435897435897436


100%|██████████| 117/117 [00:20<00:00,  5.75it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9995901584625244 - Threshold: 0.999172568321228 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9247311827956989 - Recall(box): 0.6567164179104478 - Recall(box_2): 0.8461538461538461


100%|██████████| 117/117 [00:20<00:00,  5.82it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.999172568321228 - Threshold: 0.955795168876648 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9354838709677419 - Recall(box): 0.6865671641791045 - Recall(box_2): 0.8717948717948718


100%|██████████| 117/117 [00:20<00:00,  5.77it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9093894958496094 - Threshold: 0.738972544670105 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9354838709677419 - Recall(box): 0.7313432835820896 - Recall(box_2): 0.8888888888888888


In [6]:
test_false_positive("model/tammi_oi_20251125/tammi_oi_20251125_2000k.onnx",'tammi_oi_20251125_2000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [14:15<00:00,  2.58s/it]  


Model: tammi_oi_20251125_2000k
Number of test cases: 0
Max wrong score: 0.9999980926513672
Durations: 1.4038303819444458


100%|██████████| 500/500 [05:56<00:00,  1.40it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999980926513672 - Threshold: 0.9999980926513672 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8709677419354839 - Recall(box): 0.6268656716417911 - Recall(box_2): 0.7435897435897436 - Recall(box_teamtest_silent_20251117) 0.27


100%|██████████| 500/500 [04:21<00:00,  1.92it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999033212661743 - Threshold: 0.9998852014541626 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9139784946236559 - Recall(box): 0.6567164179104478 - Recall(box_2): 0.8205128205128205 - Recall(box_teamtest_silent_20251117) 0.332


100%|██████████| 500/500 [01:10<00:00,  7.13it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9998852014541626 - Threshold: 0.9987215399742126 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9139784946236559 - Recall(box): 0.6940298507462687 - Recall(box_2): 0.8632478632478633 - Recall(box_teamtest_silent_20251117) 0.358


100%|██████████| 500/500 [01:12<00:00,  6.92it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9796262383460999 - Threshold: 0.8311673402786255 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9139784946236559 - Recall(box): 0.7611940298507462 - Recall(box_2): 0.8974358974358975 - Recall(box_teamtest_silent_20251117) 0.464


In [3]:
test_false_positive("model/tammi_oi_20251126/tammi_oi_20251126.onnx",'tammi_oi_20251126',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [17:31<00:00,  3.17s/it]  


Model: tammi_oi_20251126
Number of test cases: 0
Max wrong score: 0.9999916553497314
Durations: 1.4038303819444458


100%|██████████| 500/500 [01:27<00:00,  5.69it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999916553497314 - Threshold: 0.9999916553497314 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.8709677419354839 - Recall(box): 0.5970149253731343 - Recall(box_2): 0.7435897435897436 - Recall(box_teamtest_silent_20251117) 0.336


100%|██████████| 500/500 [01:19<00:00,  6.30it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9995901584625244 - Threshold: 0.999172568321228 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9247311827956989 - Recall(box): 0.6567164179104478 - Recall(box_2): 0.8461538461538461 - Recall(box_teamtest_silent_20251117) 0.398


100%|██████████| 500/500 [05:26<00:00,  1.53it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.999172568321228 - Threshold: 0.955795168876648 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9354838709677419 - Recall(box): 0.6865671641791045 - Recall(box_2): 0.8717948717948718 - Recall(box_teamtest_silent_20251117) 0.458


100%|██████████| 500/500 [05:33<00:00,  1.50it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9093894958496094 - Threshold: 0.738972544670105 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9354838709677419 - Recall(box): 0.7313432835820896 - Recall(box_2): 0.8888888888888888 - Recall(box_teamtest_silent_20251117) 0.482


In [21]:
test_false_positive("model/tammi_oi_20251126/tammi_oi_20251126.onnx",'tammi_oi_20251126',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:03<00:00,  1.81it/s]


Model: tammi_oi_20251126
Number of test cases: 0
Max wrong score: 0.9999916553497314
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 11.62it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9999916553497314 - Threshold: 0.9999916553497314 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9310344827586207 - Recall(box): 0.6201550387596899 - Recall(box_2): 0.7565217391304347 - Recall(box_teamtest_silent_20251117): 0.37317397078353254 - Recall(box_teamtest_noise_20251118): 0.23481781376518218 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.41it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9995901584625244 - Threshold: 0.999172568321228 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.8608695652173913 - Recall(box_teamtest_silent_20251117): 0.4555112881806109 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.27it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.999172568321228 - Threshold: 0.955795168876648 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8869565217391304 - Recall(box_teamtest_silent_20251117): 0.5139442231075697 - Recall(box_teamtest_noise_20251118): 0.3805668016194332 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.69it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9093894958496094 - Threshold: 0.738972544670105 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.5431606905710491 - Recall(box_teamtest_noise_20251118): 0.4008097165991903 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [4]:
test_false_positive("model/tammi_oi_20251202/tammi_oi_20251202.onnx",'tammi_oi_20251202',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:46<00:00,  1.16it/s]


Model: tammi_oi_20251202
Number of test cases: 0
Max wrong score: 0.999990701675415
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:01<00:00, 10.60it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.999990701675415 - Threshold: 0.999990701675415 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9425287356321839 - Recall(box): 0.6201550387596899 - Recall(box_2): 0.7652173913043478 - Recall(box_teamtest_silent_20251117): 0.30810092961487384 - Recall(box_teamtest_noise_20251118): 0.19433198380566802 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.97it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9998114109039307 - Threshold: 0.9974277019500732 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.8434782608695652 - Recall(box_teamtest_silent_20251117): 0.41832669322709165 - Recall(box_teamtest_noise_20251118): 0.2591093117408907 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.68it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9974277019500732 - Threshold: 0.9931969046592712 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6976744186046512 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.4435590969455511 - Recall(box_teamtest_noise_20251118): 0.26720647773279355 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.64it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9529258012771606 - Threshold: 0.6284633874893188 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7674418604651163 - Recall(box_2): 0.9217391304347826 - Recall(box_teamtest_silent_20251117): 0.5391766268260292 - Recall(box_teamtest_noise_20251118): 0.3684210526315789 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [6]:
def test_false_positive1(model_path, model_name, test_file_paths, start_end=[0,500]):
    oww = openwakeword.Model(
        wakeword_models=[model_path],
        enable_speex_noise_suppression=False,
        vad_threshold=0.0
    )
    outputs = []
    fpph = 0
    fn = 0
    all = 0
    durations = 0

    results = []
    for test_file_path in test_file_paths:
        for file in tqdm.tqdm(os.listdir(test_file_path)[start_end[0]:start_end[1]]):
            if file.endswith(".wav"):
                scores = oww.predict_clip(os.path.join(test_file_path,file))
                filepath = os.path.join(test_file_path, file)
                with contextlib.closing(wave.open(filepath, 'r')) as f:
                    frames = f.getnframes()
                    rate = f.getframerate()
                    duration = frames / float(rate) / 3600
                results+=[s[model_name] for s in scores]
                durations+=duration
    
    excepted_fpph = [0,1,2,3,4,5,10]
    topn = sorted(results, reverse=True)
    return topn

topn = test_false_positive1("model/tammi_oi_20251202/tammi_oi_20251202_4000k.onnx",'tammi_oi_20251202_4000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:38<00:00,  2.09it/s]


In [8]:
def test_false_positive2(model_path, model_name, test_file_paths, start_end=[0,500]):
        threshold = 0.9999993
        r1=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_1_thu_am_tu_dien_thoai_trung_tam_cong_nghe\converted_wav",threshold=threshold)
        r2=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\\",threshold=threshold)
        r3=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos_2\\",threshold=threshold)
        r4=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\\",threshold=threshold)
        r5=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test\\",threshold=threshold)
        r6=test(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251118_team_test_hard\\",threshold=threshold)

        print(f"Recall(phone): {r1} - Recall(box): {r2} - Recall(box_2): {r3} - Recall(box_teamtest_silent_20251117): {r4} - Recall(box_teamtest_noise_20251118): {r5} - Recall(box_teamtest_noise_20251118_hard): {r6}")

test_false_positive2("model/tammi_oi_20251202/tammi_oi_20251202_4000k.onnx",'tammi_oi_20251202_4000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 18/18 [00:03<00:00,  5.04it/s]

Recall(phone): 0.9655172413793104 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.39707835325365204 - Recall(box_teamtest_noise_20251118): 0.24696356275303644 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [4]:
test_false_positive("model/tammi_oi_20251202/tammi_oi_20251202_4000k.onnx",'tammi_oi_20251202_4000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:23<00:00,  1.26it/s]


Model: tammi_oi_20251202_4000k
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.96it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.76it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.70it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.72it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999999403953552 - Threshold: 0.9999992847442627 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6821705426356589 - Recall(box_2): 0.8695652173913043 - Recall(box_teamtest_silent_20251117): 0.39707835325365204 - Recall(box_teamtest_noise_20251118): 0.24696356275303644 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [3]:
test_false_positive("model/tammi_oi_20251202/tammi_oi_20251202_2000k.onnx",'tammi_oi_20251202_2000k',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:21<00:00,  2.35it/s]


Model: tammi_oi_20251202_2000k
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.47it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.22it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999998807907104 - Threshold: 0.9999996423721313 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9655172413793104 - Recall(box): 0.627906976744186 - Recall(box_2): 0.7913043478260869 - Recall(box_teamtest_silent_20251117): 0.3333333333333333 - Recall(box_teamtest_noise_20251118): 0.19838056680161945 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 10.95it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999996423721313 - Threshold: 0.9999977350234985 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9655172413793104 - Recall(box): 0.6434108527131783 - Recall(box_2): 0.8173913043478261 - Recall(box_teamtest_silent_20251117): 0.3598937583001328 - Recall(box_teamtest_noise_20251118): 0.21862348178137653 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.04it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.999819815158844 - Threshold: 0.9954031705856323 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.9043478260869565 - Recall(box_teamtest_silent_20251117): 0.4820717131474104 - Recall(box_teamtest_noise_20251118): 0.3076923076923077 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [2]:
test_false_positive("model/tammi_oi_20251205/tammi_oi_20251205.onnx",'tammi_oi_20251205',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [04:02<00:00,  1.37it/s]  


Model: tammi_oi_20251205
Number of test cases: 0
Max wrong score: 0.9990026354789734
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.38it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 0.9990026354789734 - Threshold: 0.9990026354789734 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7131782945736435 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.41434262948207173 - Recall(box_teamtest_noise_20251118): 0.26720647773279355 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.40it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9965881705284119 - Threshold: 0.9936319589614868 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7209302325581395 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.43824701195219123 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.82it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9936319589614868 - Threshold: 0.9922627210617065 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7286821705426356 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.4409030544488712 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  5.17it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.8503286838531494 - Threshold: 0.6482258439064026 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7596899224806202 - Recall(box_2): 0.9130434782608695 - Recall(box_teamtest_silent_20251117): 0.5179282868525896 - Recall(box_teamtest_noise_20251118): 0.3481781376518219 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [28]:
test_false_positive("model/tammi_oi_20251212/tammi_oi_20251212.onnx",'tammi_oi_20251212',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [03:48<00:00,  1.45it/s]


Model: tammi_oi_20251212
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458


100%|██████████| 18/18 [00:03<00:00,  5.47it/s]


With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:03<00:00,  4.56it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.50it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0.0 - Recall(box): 0.0 - Recall(box_2): 0.0 - Recall(box_teamtest_silent_20251117): 0.0 - Recall(box_teamtest_noise_20251118): 0.0 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.42it/s]

With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9999924898147583 - Threshold: 0.9979749917984009 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7441860465116279 - Recall(box_2): 0.8956521739130435 - Recall(box_teamtest_silent_20251117): 0.4435590969455511 - Recall(box_teamtest_noise_20251118): 0.2874493927125506 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [4]:
test_false_positive("model/tammi_oi_20251216/tammi_oi_20251216.onnx",'tammi_oi_20251216',
                    [r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\neg\\"])

100%|██████████| 332/332 [02:12<00:00,  2.50it/s]


Model: tammi_oi_20251216
Number of test cases: 0
Max wrong score: 1.0
Durations: 1.4038303819444458
With requirement: False Positive per hour = 0 - excepted FP = 0
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0 - Recall(box): 0 - Recall(box_2): 0 - Recall(box_teamtest_silent_20251117): 0 - Recall(box_teamtest_noise_20251118): 0 - Recall(box_teamtest_noise_20251118_hard): 0
With requirement: False Positive per hour = 1 - excepted FP = 1
Top wrong score: 1.0 - Threshold: 1.0 - FP: 0 - FPPH(box): 0.0 - Recall(phone): 0 - Recall(box): 0 - Recall(box_2): 0 - Recall(box_teamtest_silent_20251117): 0 - Recall(box_teamtest_noise_20251118): 0 - Recall(box_teamtest_noise_20251118_hard): 0


100%|██████████| 18/18 [00:01<00:00, 11.74it/s]


With requirement: False Positive per hour = 2 - excepted FP = 2
Top wrong score: 0.9999999403953552 - Threshold: 0.9999999403953552 - FP: 2 - FPPH(box): 1.4246735401393718 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6201550387596899 - Recall(box_2): 0.7739130434782608 - Recall(box_teamtest_silent_20251117): 0.3054448871181939 - Recall(box_teamtest_noise_20251118): 0.17408906882591094 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.56it/s]


...

With requirement: False Positive per hour = 4 - excepted FP = 5
Top wrong score: 0.9999816417694092 - Threshold: 0.9999768137931824 - FP: 5 - FPPH(box): 3.5616838503484294 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6589147286821705 - Recall(box_2): 0.8260869565217391 - Recall(box_teamtest_silent_20251117): 0.3545816733067729 - Recall(box_teamtest_noise_20251118): 0.21052631578947367 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.25it/s]


With requirement: False Positive per hour = 5 - excepted FP = 7
Top wrong score: 0.9999768137931824 - Threshold: 0.9999680519104004 - FP: 7 - FPPH(box): 4.986357390487801 - Recall(phone): 0.9770114942528736 - Recall(box): 0.6589147286821705 - Recall(box_2): 0.8260869565217391 - Recall(box_teamtest_silent_20251117): 0.35856573705179284 - Recall(box_teamtest_noise_20251118): 0.2145748987854251 - Recall(box_teamtest_noise_20251118_hard): 0.0


100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


With requirement: False Positive per hour = 10 - excepted FP = 14
Top wrong score: 0.9992738962173462 - Threshold: 0.9825006127357483 - FP: 14 - FPPH(box): 9.972714780975602 - Recall(phone): 0.9770114942528736 - Recall(box): 0.7286821705426356 - Recall(box_2): 0.8782608695652174 - Recall(box_teamtest_silent_20251117): 0.43691899070385126 - Recall(box_teamtest_noise_20251118): 0.27530364372469635 - Recall(box_teamtest_noise_20251118_hard): 0.0


In [ ]:
"tam miơi",
"tammi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",
"tam mi ơi",

In [9]:
outputs = get_wrong_answer("model/old/tammi_oi_20251016_17_20_23_20251103_raw/tammi_oi_20251016_17_20_23_20251103_raw_2.onnx",'tammi_oi_20251016_17_20_23_20251103_raw_2',threshold=0.738972544670105)

100%|██████████| 753/753 [00:57<00:00, 13.08it/s]


In [10]:
outputs[0]

['duhv2_WhatsApp_Audio_2025-10-07_at_08.35.25.wav',
 'ngocvtb_WhatsApp_Audio_2025-10-07_at_11.13.46.wav',
 'ngocvtb_WhatsApp_Audio_2025-10-07_at_11.32.09.wav',
 'nnquang_WhatsApp_Ptt_2025-10-07_at_09.25.37.wav',
 'QuangHoang_942434289_WhatsApp_Audio_2025-10-07_at_09.56.42.wav',
 'trangdtt1@is_WhatsApp_Audio_2025-10-07_at_09.55.54.wav',
 'trangdtt1@is_WhatsApp_Audio_2025-10-07_at_09.55.55.wav',
 'trangdtt1@is_WhatsApp_Audio_2025-10-07_at_09.55.55_(1).wav',
 'trangdtt1@is_WhatsApp_Audio_2025-10-07_at_09.55.56.wav',
 'trangpvt_WhatsApp_Ptt_2025-10-07_at_09.30.13.wav',
 'trangpvt_WhatsApp_Ptt_2025-10-07_at_09.31.40.wav',
 'trangpvt_WhatsApp_Ptt_2025-10-07_at_09.31.56.wav',
 'TranQuangHuy_0325727358_WhatsApp_Audio_2025-10-07_at_09.50.30.wav']

In [ ]:
def get_wrong_answer(model_path, model_name, threshold):
    r1=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_1_thu_am_tu_dien_thoai_trung_tam_cong_nghe\converted_wav",threshold=threshold)
    r2=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\\",threshold=threshold)
    r3=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos_2\\",threshold=threshold)
    r4=test_wrong_answer(model_path,model_name,r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\\",threshold=threshold)
    return [r1,r2,r3,r4]

In [13]:
for i in outputs[1]:
    print(os.path.join(r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos",i))

C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\0_recorded_audio_failed_5 (mp3cut.net).wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\10_recorded_audio_failed_5 (mp3cut.net).wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\2_recorded_audio_24_09_25 (mp3cut.net).wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\2_recorded_audio_failed_5 (mp3cut.net).wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\5_recorded_audio_failed_5 (mp3cut.net).wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\pos\6_recorded_audio_failed_5 (mp3cut.net).wav
C:\Users\User\D

In [15]:
for i in outputs[3]:
    print(os.path.join(r"C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test",i))

C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\audio_raw_20250822_215053_002_23.564487_24.130200.wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\audio_raw_20250822_215053_004_108.539612_109.377895.wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\audio_raw_20250822_215053_006_124.256128_125.495552.wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\audio_raw_20250822_215053_009_150.201749_151.379459.wav
C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\all_test_verify_checkpoints\test_11_thu_am_tu_box\20251117_team_test\audio_raw_20250822_215053_011_173.329095_174.511948.wav
C:\Users\User\Documents\Workspace\Code\pyt

In [ ]:
outputs = {
    'my_model_p2_11_v4.xlsx':''
    ,'tammi_new_model2.xlsx':''
    ,'tammi_oi_20251016.xlsx':50000
    ,'tammi_oi_20251016_20251017.xlsx':50000
    ,'tammi_oi_20251016_17_20.xlsx':100000
    ,'tammi_oi_20251016_17_20_22.xlsx':100000
    ,'tammi_oi_20251016_17_20_22_150ks.xlsx':150000
    ,'tammi_oi_20251016_17_20_22_100ks_50ks.xlsx':150000
    ,'tammi_oi_20251016_17_20_23.xlsx':200000
    ,'tammi_oi_20251016_17_20_31.xlsx':300000
    ,'tammi_oi_20251031.xlsx':200000
}

In [192]:
all_df = pd.read_excel(r'C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\test_verify_checkpoints\test_checkpoints_doc\\'+list(outputs.keys())[0])
all_df['Epoch'] = [outputs[list(outputs.keys())[0]] for i in range(len(all_df))]
all_df['Phase'] = ['old' for i in range(len(all_df))]
all_df['STT'] = [1 for i in range(len(all_df))]
for e,k in enumerate(list(outputs.keys())[1:]):
    df = pd.read_excel(r'C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\test_verify_checkpoints\test_checkpoints_doc\\'+k)
    df['Epoch'] = [outputs[k] for i in range(len(df))]
    df['Phase'] = ['new' if outputs[k]!='' else 'old' for i in range(len(df))]
    df['STT'] = [e+2 for i in range(len(df))]
    all_df = pd.concat([all_df,df])

In [193]:
all_df['Recall'] = (all_df['Recall (phone)']*93 + all_df['Recall (box)']*134 + all_df['Recall (box_2)']*117)/(93+134+117)

In [194]:
all_df.to_excel(r'C:\Users\User\Documents\Workspace\Code\python\speech\tammi_oi\data\test_verify_checkpoints\test_checkpoints_doc\report.xlsx',index=False)

In [195]:
all_df

,STT,Model,Expected FPPH,Threshold,FPPH,Recall (phone),Recall (box),Recall (box_2),Epoch,Phase,Recall
0,1,my_model_p2_11_v4,0,0.940588,0.000000,0.430108,0.432836,0.427350,,old,0.430233
1,1,my_model_p2_11_v4,1,0.715262,0.712337,0.430108,0.462687,0.427350,,old,0.441860
2,1,my_model_p2_11_v4,2,0.674800,1.424674,0.430108,0.462687,0.427350,,old,0.441860
3,1,my_model_p2_11_v4,3,0.033564,2.849347,0.494624,0.537313,0.461538,,old,0.500000
4,1,my_model_p2_11_v4,4,0.029682,3.561684,0.505376,0.537313,0.470085,,old,0.505814
...,...,...,...,...,...,...,...,...,...,...,...
2,9,tammi_oi_20251016_17_20_23,2,0.999982,1.424674,0.870968,0.694030,0.820513,200000,new,0.784884
3,9,tammi_oi_20251016_17_20_23,3,0.999892,2.849347,0.881720,0.716418,0.854701,200000,new,0.808140
4,9,tammi_oi_20251016_17_20_23,4,0.999860,3.561684,0.881720,0.716418,0.854701,200000,new,0.808140
5,9,tammi_oi_20251016_17_20_23,5,0.999309,4.986357,0.881720,0.746269,0.888889,200000,new,0.831395


In [1]:
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

  Attempting uninstall: speechbrain
    Found existing installation: speechbrain 1.0.3
    Uninstalling speechbrain-1.0.3:
      Successfully uninstalled speechbrain-1.0.3

  Attempting uninstall: librosa

    Found existing installation: librosa 0.11.0

    Uninstalling librosa-0.11.0:

      Successfully uninstalled librosa-0.11.0

  Attempting uninstall: audiomentations

    Found existing installation: audiomentations 0.43.1

   -------------------- ------------------- 1/2 [audiomentations]
    Uninstalling audiomentations-0.43.1:
   -------------------- ------------------- 1/2 [audiomentations]
      Successfully uninstalled audiomentations-0.43.1
   -------------------- ------------------- 1/2 [audiomentations]
   ---------------------------------------- 2/2 [audiomentations]

  Attempting uninstall: torch-audiomentations
    Found existing installation: torch-audiomentations 0.12.0
    Uninstalling torch-audiomentations-0.12.0:
      Successfully uninstalled torch-audiomentation

ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.8.1 (from versions: 2.12.0rc0, 2.12.0rc1, 2.12.0, 2.12.1, 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow-cpu==2.8.1


   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.3 MB 645.7 kB/s eta 0:00:09
   ---- ----------------------------------- 0.8/6.3 MB 817.9 kB/s eta 0:00:07
   ------ --------------------------------- 1.0/6.3 MB 825.2 kB/s eta 0:00:07
   -------- ------------------------------- 1.3/6.3 MB 945.5 kB/s eta 0:00:06
   --------- ------------------------------ 1.6/6.3 MB 999.0 kB/s eta 0:00:05
   ----------- ---------------------------- 1.8/6.3 MB 1.0 MB/s eta 0:00:05
   ------------- -------------------------- 2.1/6.3 MB 1.1 MB/s eta 0:00:04
   -------------- ------------------------- 2.4/6.3 MB 1.1 MB/s eta 0:00:04
   -------------- ------------------------- 2.4/6.3 MB 1.1 MB/s eta 0:00:04
   -------------- ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.16 requires torchvision, which is not installed.
dvc 3.61.0 requires fsspec>=2024.2.0, but you have fsspec 2023.10.0 which is incompatible.
dvc-data 3.16.10 requires fsspec>=2024.2.0, but you have fsspec 2023.10.0 which is incompatible.
dvc-objects 5.1.1 requires fsspec>=2024.2.0, but you have fsspec 2023.10.0 which is incompatible.
s3fs 2025.7.0 requires fsspec==2025.7.0, but you have fsspec 2023.10.0 which is incompatible.
scmrepo 3.4.0 requires fsspec[tqdm]>=2024.2.0, but you have fsspec 2023.10.0 which is incompatible.


  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for deep-phonemizer: filename=deep_phonemizer-0.0.19-py3-none-any.whl size=33377 sha256=679ee24b34c493b7e6f5bb096337fb009ad26324ac0a06831f572245300d3b6e
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\73\cc\01\1c74a1f4e6ba31a42bb82f4e3d852e2f23236fe3e5d589dcf3
Successfully built deep-phonemizer


  DEPRECATION: Building 'deep-phonemizer' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'deep-phonemizer'. Discussion can be found at https://github.com/pypa/pip/issues/6334
